sec7B — numbers quoted in Sec. VII B (lossy plaquette) and the captions of Figs. `fig:plaq`, `fig:zeno`
Re-derived from the shipped data of `tab_plaqerr`, `fig_plaq`, `fig_zeno` and of this directory (exact runs at
gamma/J = 50, 100). The table cells themselves are checked in `tab_plaqerr.ipynb`.

In [1]:
import sys
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
RUNS = IO.read_data(HERE.parent / "tab_plaqerr" / "data")["runs.csv"]
EXT = IO.read_data(HERE.parent / "tab_plaqerr" / "data")["exact.csv"]
CUR = IO.read_data(HERE.parent / "fig_plaq" / "data")["curves.csv"]
Z = IO.read_data(HERE.parent / "fig_zeno" / "data"); K, WD = Z["kappa.csv"], Z["window.csv"]
KB = IO.read_data(HERE / "data")["kappa_biggamma.csv"]
ROWS = []
def check(where, printed, s):
    ROWS.append((where, printed, s, s == printed))
run = lambda **w: IO.select(RUNS, **w)
one = lambda tab, q, **w: IO.select(tab, **w)[q][0]
pct = lambda v: f"{100 * v:.0f}\\%"

In [2]:
# dark states
v = run(ic="3u1d"); ok = v["method"] != "mGW"
check("3u1d: N(200) of all schemes except mGW, max |N - 2|", "$N_\\infty = 2$",
      "$N_\\infty = 2$" if np.abs(v["N_end"][ok] - 2).max() < 5e-4 else f"{np.abs(v['N_end'][ok] - 2).max():.1e}")
check("3u1d: exact N(200), all gamma", "2", "2" if np.abs(IO.select(EXT, ic="3u1d")["N_end"] - 2).max() < 5e-4 else "?")
check("mixed, gamma/J = 3: exact N_inf", "1.53", f"{one(K, 'N_inf', method='exact', ic='mixed', gamma=3.0):.2f}")
check("mixed, gamma/J = 3: DSL N_inf", "1.46", f"{one(K, 'N_inf', method='DSL', ic='mixed', gamma=3.0):.2f}")
check("mixed, gamma/J = 3: 2B N(200)", "1.65", f"{one(RUNS, 'N_end', method='2B', ic='mixed', gamma=3.0):.2f}")
check("mixed, gamma/J = 3: mGW N(200)", "4", f"{one(RUNS, 'N_end', method='mGW', ic='mixed', gamma=3.0):.0f}")
for m in ("HF", "TPPX"):
    v = IO.select(CUR, method=m, ic="mixed", gamma=3.0)["N"]
    check(f"mixed, gamma/J = 3: {m} decays toward zero (N(200) < 0.1, N decreasing after tJ = 10)", "-> 0",
          "-> 0" if v[-1] < 0.1 and np.all(np.diff(v[100:]) <= 1e-12) else f"{v[-1]:.3f}")
check("DSL <S^2> drift, max over the 35 runs (conserved: below the integration tolerance 1e-6)", "conserved",
      "conserved" if run(method='DSL')['S2_drift'].max() < 1e-6 else f"{run(method='DSL')['S2_drift'].max():.0e}")
# strong loss
check("Slater, gamma/J = 10: exact N_inf (N(200))", "0.126", f"{one(EXT, 'N_end', ic='slater', gamma=10.0):.3f}")
check("Slater, gamma/J = 10: DSL N(200)", "0.0066", f"{one(RUNS, 'N_end', method='DSL', ic='slater', gamma=10.0):.4f}")
for m in ("2B", "mGW"):
    c = IO.select(CUR, method=m, ic="slater", gamma=10.0)
    check(f"Slater, gamma/J = 10: {m} gains particles after tJ ~ (time of min N)", "0.2", f"{c['t'][np.argmin(c['N'])]:.1f}")
G5 = [0.1, 0.3, 1.0, 3.0, 10.0]
dfc = [1 - one(RUNS, "N_end", method="DSL", ic="slater", gamma=g) / one(EXT, "N_end", ic="slater", gamma=g) for g in G5]
check("Slater: DSL deficit at tJ = 200, gamma/J = 0.1", "24\\%", pct(dfc[0]))
check("Slater: DSL deficit at tJ = 200, gamma/J = 10", "95\\%", pct(dfc[-1]))
check("Slater: deficit grows monotonically", "monotonic", "monotonic" if np.all(np.diff(dfc) > 0) else "no")
# double occupancy
for ic, g in (("one_pair", 1.0), ("cdw", 3.0)):
    check(f"DSL mean |D - D_ex| / D(0), tJ <= 20, {ic} gamma/J = {g:g}", "$3\\times10^{-4}$",
          f"$" + f"{one(RUNS, 'err_D', method='DSL', ic=ic, gamma=g):.0e}".replace("e-0", "\\times10^{-") + "}$")
    c = IO.select(CUR, method="DSL", ic=ic, gamma=g)
    check(f"DSL keeps D >= 0, {ic} gamma/J = {g:g} (min D, t > 0)", "D >= 0", "D >= 0" if c["D"][1:].min() >= 0 else f"{c['D'][1:].min():.1e}")
pos = np.isfinite(RUNS["err_D"]) | ((RUNS["status"] != "Ok") & (RUNS["ic"] != "neel"))
cases = sorted({(i, g) for i, g, p in zip(RUNS["ic"], RUNS["gamma"], pos) if p})
check("runs with D(0) > 0", "30", f"{len(cases)}")
best = 0
for ic, g in cases:
    s = run(ic=ic, gamma=g); e = np.where(np.isfinite(s["err_D"]), s["err_D"], np.inf)
    best += s["method"][np.argmin(e)] == "DSL"
check("... in which the DSL error in D(t) is the smallest", "30", f"{best}")
for m, p in (("2B", "$-0.03$"), ("mGW", "$-1.6$")):
    v = min(one(RUNS, "D_min", method=m, ic="one_pair", gamma=1.0), one(RUNS, "D_min", method=m, ic="cdw", gamma=3.0))
    check(f"{m}: min D in Fig. plaq(c,d)", p, f"${v:.{2 if m == '2B' else 1}f}$")

In [3]:
# Zeno crossover
for ic, g, p in (("mixed", 50.0, "28"), ("mixed", 100.0, "29"), ("neel", 50.0, "18"), ("neel", 100.0, "18")):
    check(f"exact gamma kappa / J^2, {ic}, gamma/J = {g:g}", p, f"{one(KB, 'gamma_kappa', ic=ic, gamma=g):.0f}")
kz = lambda m, ic: (lambda s: (s["gamma"], s["kappa"]))(IO.select(K, method=m, ic=ic))
for ic in ("mixed", "neel"):
    g, k = kz("exact", ic)
    check(f"exact kappa peak position, {ic}", "2--3", "2--3" if g[np.nanargmax(k)] in (2.0, 3.0) else f"{g[np.nanargmax(k)]:g}")
g, ke = kz("exact", "mixed"); _, kd = kz("DSL", "mixed"); s = g <= 10
check("mixed: DSL kappa below exact for gamma/J <= 10, smallest deficit", "13", f"{100 * (1 - kd / ke)[s].min():.0f}")
check("mixed: DSL kappa below exact for gamma/J <= 10, largest deficit", "20", f"{100 * (1 - kd / ke)[s].max():.0f}")
_, kh = kz("HF", "mixed")
check("mixed: MF kappa / gamma at gamma/J = 30", "0.4", f"{(kh / g)[g == 30.0][0]:.1f}")
check("mixed: MF kappa grows monotonically", "monotonic", "monotonic" if np.all(np.diff(kh) > 0) else "no")
w = lambda m, ic, g, q="D_over_O": one(WD, q, method=m, ic=ic, gamma=g)
check("mixed: TPPX window D/O at gamma/J = 30 (levels off)", "0.36", f"{w('TPPX', 'mixed', 30.0):.2f}")
check("Neel: TPPX window D/O at gamma/J = 30", "0.87", f"{w('TPPX', 'neel', 30.0):.2f}")
g, ke = kz("exact", "neel")
for m, p in (("HF", "43\\%"), ("TPPX", "63\\%")):
    _, km = kz(m, "neel"); check(f"Neel: {m} underestimates the kappa peak by", p, pct(1 - np.nanmax(km) / np.nanmax(ke)))
_, kd = kz("DSL", "neel"); s = g <= 7
check("Neel: DSL max |kappa/kappa_ex - 1| for gamma/J <= 7", "12\\%", pct(np.abs(kd / ke - 1)[s].max()))
st = IO.select(K, method="DSL", ic="neel")
check("Neel: DSL stops (diverges) for gamma/J >= 10", "10", f"{st['gamma'][st['status'] != 'Ok'].min():g}")
check("plaquette trajectories in which the DSL diverges (of 35)", "1", f"{(run(method='DSL')['status'] != 'Ok').sum()}")

In [4]:
# positivity (threshold 1e-6 on D_r and the eigenvalues of rho; lower bounds)
viol = lambda s: (s["D_site_min"] < -1e-6) | (s["eig_min"] < -1e-6)
for m, p in (("HF", "0/35"), ("TPPX", "3/35"), ("DSL", "14/35"), ("2B", "25/35"), ("mGW", "35/35")):
    s = run(method=m); check(f"{m}: trajectories violating positivity", p, f"{viol(s).sum()}/{len(s['gamma'])}")
s = run(method="TPPX"); v = viol(s)
check("TPPX violations all from the single pair start", "one_pair", ",".join(sorted(set(s["ic"][v]))))
check("TPPX min D_r", "$-1.5\\times10^{-4}$", "$" + f"{s['D_site_min'].min():.1e}".replace("e-0", "\\times10^{-") + "}$")
s = run(method="DSL"); ok = s["status"] == "Ok"
check("DSL min D_r (excluding the diverged run)", "$-8.1\\times10^{-4}$",
      "$" + f"{s['D_site_min'][ok].min():.1e}".replace("e-0", "\\times10^{-") + "}$")
i = np.argmin(np.where(ok, s["eig_min"], np.inf))
check("DSL min eigenvalue of rho (excluding the diverged run)", "$-0.014$", f"${s['eig_min'][i]:.3f}$")
check("... from the start", "3u1d", s["ic"][i])
s = run(method="mGW"); ok = s["status"] == "Ok"
check("mGW min eigenvalue of rho (non-diverged runs)", "$-0.49$", f"${s['eig_min'][ok].min():.2f}$")
check("mGW min eigenvalue of rho in its diverged run", "$-1.05$", f"${s['eig_min'][~ok].min():.2f}$")
s = run(method="DSL"); ok = s["status"] == "Ok"; pq = np.minimum(np.minimum(s["P_min"], s["Q_min"]), s["Gp_min"])
check("DSL runs with a P/Q/G' eigenvalue below -1e-6 (non-diverged)", "30 of 34", f"{(pq[ok] < -1e-6).sum()} of {ok.sum()}")
check("DSL min P/Q/G' eigenvalue (non-diverged)", "$-0.06$", f"${pq[ok].min():.2f}$")

In [5]:
w = max(len(r[0]) for r in ROWS)
print(f"{'quantity':{w}s}  {'paper':22s} {'new':22s} match")
for r in ROWS:
    print(f"{r[0]:{w}s}  {r[1]:22s} {r[2]:22s} {'yes' if r[3] else 'NO'}")
print(f"\n{sum(r[3] for r in ROWS)} of {len(ROWS)} match")

quantity                                                                                 paper                  new                    match
3u1d: N(200) of all schemes except mGW, max |N - 2|                                      $N_\infty = 2$         $N_\infty = 2$         yes
3u1d: exact N(200), all gamma                                                            2                      2                      yes
mixed, gamma/J = 3: exact N_inf                                                          1.53                   1.53                   yes
mixed, gamma/J = 3: DSL N_inf                                                            1.46                   1.46                   yes
mixed, gamma/J = 3: 2B N(200)                                                            1.65                   1.65                   yes
mixed, gamma/J = 3: mGW N(200)                                                           4                      4                      yes
mixed, gamma/J = 3: HF de